# Register publications in entitycore
Note: Requires admin access atm

Publication: Shiu, Sterne et al. (2024), *A Drosophila computational brain model reveals sensorimotor processing*, **Nature**. DOI: [10.1038/s41586-024-07763-9](https://www.nature.com/articles/s41586-024-07763-9).

Reads from the local spreadsheet `Drosophila_brain_model_publication.xlsx`.

In [ ]:
import pandas as pd
import os

from datetime import datetime
from entitysdk import Client, ProjectContext, models
from obi_auth import get_token

In [ ]:
# Staging environment
token = get_token(environment="staging")
project_context = ProjectContext.from_vlab_url("https://staging.openbraininstitute.org/app/virtual-lab/lab/1f91f30e-1489-4e2a-8eb7-1217257c8e19/project/7a411785-6895-4839-aaa2-d9f76e09875a/home")
client = Client(environment="staging", project_context=project_context, token_manager=token)

# Production environment
# token = get_token(environment="production")
# project_context = ProjectContext.from_vlab_url("https://www.openbraininstitute.org/app/virtual-lab/lab/5f8376bf-b84f-4188-8ef5-e1df3d7529b4/project/7d22829c-edc6-4b1d-8ab9-99dd9e511e74/home")
# client = Client(environment="production", project_context=project_context, token_manager=token)

In [ ]:
# Spreadsheet with the publication(s) to be registered.
# Single paper: DOI 10.1038/s41586-024-07763-9
publications_file = "Drosophila_brain_model_publication.xlsx"  # relative to this notebook's folder

In [ ]:
check_only = True  # If True, only check, no registration

### Load list of publications to be registered

In [ ]:
publications = pd.read_excel(publications_file, sheet_name="publications")
print(f"{publications.shape[0]} publication(s) in '{publications_file}'")

### Register publications

In [ ]:
def remove_multiple_spaces(text):
    split = text.strip().split(" ")
    split = [_txt for _txt in split if len(_txt) > 0]
    return " ".join(split)

def remove_dots(text):
    return text.replace(".", "")

def get_author_list(authors):
    """Split list of authors into individual given/familiy name strings.

    Assuming the following format:
    "FirstName(s)1, FamilyName1; FirstName(s)2, FamiliyName2; ..."
    """
    raw_names_list = remove_dots(authors).split(";")
    names_list = []
    for _name in raw_names_list:
        split = _name.split(",")
        assert len(split) == 2, "ERROR: Wrong format, only one comma expected!"
        author = {"given_name": remove_multiple_spaces(split[0]),
                  "family_name": remove_multiple_spaces(split[1])}
        names_list.append(author)
    return names_list

In [ ]:
for idx, row in publications.iterrows():
    author_list = get_author_list(row.authors)

    # Create model
    publ_model = models.Publication(
        DOI=row["doi"].strip(),
        title=row["title"].strip(),
        authors=author_list,
        publication_year=row["publicationDate"].year,
        abstract=row["abstract"].strip(),
    )

    # Register new entity
    res = client.search_entity(entity_type=models.Publication, query={"DOI": publ_model.DOI}).all()
    if len(res) == 0:
        if check_only:
            print(f"***CHECK ONLY*** {publ_model}")
        else:
            registered_publ = client.register_entity(publ_model)
    else:
        print(f"***SKIPPING*** Publication with DOI {publ_model.DOI} already registered")